In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Lambda

with open("CBOW/document3.txt", "r") as file:
    text = file.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

words = tokenizer.texts_to_sequences([text])[0]

print("Total words:", len(words))
print("Vocabulary size:", len(tokenizer.word_index))

context = []
target = []

window = 2

for i in range(window, len(words) - window):
    context.append(words[i-window:i] + words[i+1:i+window+1])
    target.append(words[i])

context = np.array(context)
target = to_categorical(target, num_classes=len(tokenizer.word_index) + 1)

print("Context shape:", context.shape)
print("Target shape:", target.shape)

print(text)

model = Sequential([
    Embedding(len(tokenizer.word_index) + 1, 100),
    tf.keras.layers.GlobalAveragePooling1D(),
    Dense(len(tokenizer.word_index) + 1, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

model.fit(context, target, epochs=100, verbose=1)


loss, accuracy = model.evaluate(context, target)

print("Loss:", loss)
print("Accuracy:", accuracy)

print("CBOW Model Output:")

predictions = model.predict(context[:5])

for i in range(5):
    predicted_word = tokenizer.index_word[np.argmax(predictions[i])]
    actual_word = tokenizer.index_word[np.argmax(target[i])]

    print("Actual:", actual_word, "| Predicted:", predicted_word)


embeddings = model.layers[0].get_weights()[0]

word = "influenza"
index = tokenizer.word_index[word]

print("Embedding of", word, ":")
print(embeddings[index][:10])



